In [33]:
import sys
sys.path.append('desc_utils')

from desc_utils import  atom_numbering as an
import pandas as pd
import orca_descriptors as od
import os
import numpy as np

In [34]:
#Provide path to directory or directory of directies containing xyz files
path = "/home/aconner/orca_descriptors/library_pyox_2026_03_03/library_pyox_2026_03_03" 


In [35]:
#Define lists for decriptor retrieval 
atoms = ["H1", "H2", "C1", "C2", "N1", "N2", "Ni"]
bonds = [("C1", "N1"), ("N1", "C1"), ("C2", "N2"), ("N2", "C2"), ("Ni", "N1"), ("Ni", "N2"), ("Ni", "H1"), ("Ni", "H2")] #if order matters for 2-atom descriptors (e.g. Sterimol B1), atoms must be listed in desired order
angles = [("H2", "Ni", "H1")]
dihedrals = [("N1", "C1", "C2", "N2"), ("C2", "N2", "Ni", "H2"), ("C1", "N1", "Ni", "H1")]

#quadrants = [[2, 1, 3, 4, 6]] 


In [36]:
#generate descriptor dictionary for each row in dataframe and add to dataframe
def get_descriptors(df):
    results_dict = {}
    for row in df.itertuples():

        P = od.PropertiesXYZ(path=df.loc[row.Index, "path"])
        
        #Zero atom properties
        P.get_SASA_and_Volume()

        #One atom properties
        atom_list = []
        for atom in atoms:
            atom_list.append(int(df.loc[row.Index, atom]))
        
        P.define_atom_list(atom_list)
        P.get_Vbur()
        P.get_atomic_SASA()

        #Two atom properties
        bond_list = []
        for bond in bonds: 
            bond_list.append([int(df.loc[row.Index, bond[0]]), int(df.loc[row.Index, bond[1]])])
        P.define_bond_list(bond_list)
        P.get_bond_distances()
        P.get_Sterimol_parameters(morfeus=True)

        #Three atom properties
        angle_list = []
        for angle in angles:
            angle_list.append([int(df.loc[row.Index, angle[0]]), int(df.loc[row.Index, angle[1]]), int(df.loc[row.Index, angle[2]])])
        P.define_angle_list(angle_list)
        P.get_angles()

        #Four atom properties
        dihedral_list = []
        for dihedral in dihedrals:
            dihedral_list.append([int(df.loc[row.Index, dihedral[0]]), int(df.loc[row.Index, dihedral[1]]), int(df.loc[row.Index, dihedral[2]]), int(df.loc[row.Index, dihedral[3]])])
        P.define_dihedral_list(dihedral_list)
        P.get_dihedrals()
        #P.get_pyramidalizations()

        #Quadrant properties
        #quadrant_list = quadrants
        #P.define_quadrant_list(quadrant_list)
        #P.get_vbur_quadrants()
        #P.get_vbur_octants()

        results_dict[row.Index] = P.results
    return results_dict



In [37]:
def make_dict_atoms(df):
    dict_atoms = {}
    for row in df.itertuples():
        dict_atoms[row.Index] = {}
        for col in df.columns:
            if col.startswith("H1"):
                atom = col[:-1] 
                number = df.at[row.Index, col]
                dict_atoms[row.Index][col] = atom + str(number)
            elif col.startswith("H2"):
                atom = col[:-1] 
                number = df.at[row.Index, col]
                dict_atoms[row.Index][col] = atom + str(number)
            elif col.startswith("C1"):
                atom = col[:-1] 
                number = df.at[row.Index, col]
                dict_atoms[row.Index][col] = atom + str(number)
            elif col.startswith("C2"):
                atom = col[:-1] 
                number = df.at[row.Index, col]
                dict_atoms[row.Index][col] = atom + str(number)
            elif col.startswith("N1"):
                atom = col[:-1] 
                number = df.at[row.Index, col]
                dict_atoms[row.Index][col] = atom + str(number)
            elif col.startswith("N2"):
                atom = col[:-1] 
                number = df.at[row.Index, col]
                dict_atoms[row.Index][col] = atom + str(number)
            elif col.startswith("Ni"):
                atom = col  
                number = df.at[row.Index, col]
                dict_atoms[row.Index][col] = atom + str(number)
    return dict_atoms


In [38]:
#alter key in results dictionary to match format of user defined lists and add to dataframe
#Have to add new descriptors to this code!!
def alter_results_dict(results_dict, dict_atoms):
    new_results = {}
    for key in results_dict.keys():
        new_results[key] = {}
        for subkey in results_dict[key].keys():
            if subkey.startswith("Bond_"):
                for bond in bonds:
                    bond_key = [dict_atoms[key][bond[0]], dict_atoms[key][bond[1]]]
                    if all(bk in subkey for bk in bond_key):
                        new_key = "Bond_" + bond[0] + "_" + bond[1]
                        new_results[key][new_key] = results_dict[key][subkey]
            elif subkey.startswith("Sterimol_") and subkey.endswith("_L"):
                for bond in bonds:
                    bond_key = [dict_atoms[key][bond[0]], dict_atoms[key][bond[1]]]
                    if all(bk in subkey for bk in bond_key):
                        new_key = "Sterimol_" + bond[0] + "_" + bond[1] +"_L"
                        new_results[key][new_key] = results_dict[key][subkey]
            elif subkey.startswith("Sterimol_") and subkey.endswith("_Bmin"):
                for bond in bonds:
                    bond_key = [dict_atoms[key][bond[0]], dict_atoms[key][bond[1]]]
                    if all(bk in subkey for bk in bond_key):
                        new_key = "Sterimol_" + bond[0] + "_" + bond[1] +"_Bmin"
                        new_results[key][new_key] = results_dict[key][subkey]
            elif subkey.startswith("Sterimol_") and subkey.endswith("_Bmax"):
                for bond in bonds:
                    bond_key = [dict_atoms[key][bond[0]], dict_atoms[key][bond[1]]]
                    if all(bk in subkey for bk in bond_key):
                        new_key = "Sterimol_" + bond[0] + "_" + bond[1] +"_Bmax"
                        new_results[key][new_key] = results_dict[key][subkey]
            elif subkey.startswith("Angle_"):
                for angle in angles:
                    angle_key = [dict_atoms[key][angle[0]], dict_atoms[key][angle[1]], dict_atoms[key][angle[2]]]
                    if all(ak in subkey for ak in angle_key):
                        new_key = "Angle_" + angle[0] + "_" + angle[1] + "_" + angle[2]
                        new_results[key][new_key] = results_dict[key][subkey]
            elif subkey.startswith("Dihed_"):
                for dihedral in dihedrals:
                    dihedral_key = [dict_atoms[key][dihedral[0]], dict_atoms[key][dihedral[1]], dict_atoms[key][dihedral[2]], dict_atoms[key][dihedral[3]]]
                    if all(dk in subkey for dk in dihedral_key):
                        new_key = "Dihed_" + dihedral[0] + "_" + dihedral[1] + "_" + dihedral[2] + "_" + dihedral[3]
                        new_results[key][new_key] = results_dict[key][subkey]
            elif subkey.startswith("%Vbur_atom_"):
                for atom in atoms:
                    atom_key = dict_atoms[key][atom]
                    if atom_key in subkey:
                        new_key = "%Vbur_" + atom
                        new_results[key][new_key] = results_dict[key][subkey]
            elif subkey.startswith("SASA_"):
                for atom in atoms:
                    atom_key = dict_atoms[key][atom]
                    if atom_key in subkey:
                        new_key = "SASA_" + atom
                        new_results[key][new_key] = results_dict[key][subkey]
            else:
                new_results[key][subkey] = results_dict[key][subkey]
    return new_results



In [39]:
#add new results to dataframe
def results_to_df(df, new_results):
    for key in new_results.keys():
        for subkey in new_results[key].keys():
            df.at[key, subkey] = new_results[key][subkey]
    return df

In [40]:
def descriptor_stats(df):
    #Find min, max, and average of descriptors across dataset
    descriptor_stats = {}

    #boltzmann weighted average
    #convert energy to relative energy
    if 'Relative_energy' not in df.columns:
        df.insert(5, 'Relative_energy', df['SCF_energy'] - df['SCF_energy'].min())
    else:
        df['Relative_energy'] = df['SCF_energy'] - df['SCF_energy'].min()


    T = 298.15 #temperature in K
    R = 3.1668105e-6 #Boltzmann constant in eH/K
    weights = []

    for row in df.itertuples():
        energy = row.Relative_energy
        weight = np.exp(-energy/(R*T))
        weights.append(weight)

    normalized_weights = weights / np.sum(weights)

    for column in df.iloc[:, 15:]:
        descriptor_stats[f'{column}_min'] = df[column].min()
        descriptor_stats[f'{column}_max'] = df[column].max()
        descriptor_stats[f'{column}_mean'] = np.average(df[column], weights=normalized_weights)

    descriptor_stats_df = pd.DataFrame(descriptor_stats, index=[0])
    if df["smiles"].nunique() == 1:
        descriptor_stats_df.insert(0, "SMILES", df["smiles"].iloc[0])
        descriptor_stats_df.insert(1, "INCHI", df["inchi"].iloc[0])
    else:
        print("Multiple SMILES strings found in dataset. Please seperate before calculating descriptor statistics.")    

    return descriptor_stats_df


In [41]:
total_df = pd.DataFrame()

with os.scandir(path) as entries:
    for entry in entries:
        if entry.is_dir():
            sub_path = os.path.join(path, entry.name)
            if not sub_path[-1] == '/':
                sub_path = sub_path + '/'
            df = an.compile_final(sub_path)
            dict_atoms = make_dict_atoms(df)
            results_dict = get_descriptors(df)
            new_results = alter_results_dict(results_dict, dict_atoms)
            df = results_to_df(df, new_results)
            descriptor_stats_df = descriptor_stats(df)
            total_df = pd.concat([total_df, descriptor_stats_df], ignore_index=True)

total_df

No xyz.json file found in the specified path. Results dictionary will be empty.
No xyz.json file found in the specified path. Results dictionary will be empty.
No xyz.json file found in the specified path. Results dictionary will be empty.
No xyz.json file found in the specified path. Results dictionary will be empty.
No xyz.json file found in the specified path. Results dictionary will be empty.
No xyz.json file found in the specified path. Results dictionary will be empty.
No xyz.json file found in the specified path. Results dictionary will be empty.
No xyz.json file found in the specified path. Results dictionary will be empty.
No xyz.json file found in the specified path. Results dictionary will be empty.
No xyz.json file found in the specified path. Results dictionary will be empty.
No xyz.json file found in the specified path. Results dictionary will be empty.
No xyz.json file found in the specified path. Results dictionary will be empty.
No xyz.json file found in the specified 

,SMILES,INCHI,SASA_min,SASA_max,SASA_mean,%Vbur_H1_min,%Vbur_H1_max,%Vbur_H1_mean,%Vbur_H2_min,%Vbur_H2_max,...,Angle_H2_Ni_H1_mean,Dihed_N1_C1_C2_N2_min,Dihed_N1_C1_C2_N2_max,Dihed_N1_C1_C2_N2_mean,Dihed_C2_N2_Ni_H2_min,Dihed_C2_N2_Ni_H2_max,Dihed_C2_N2_Ni_H2_mean,Dihed_C1_N1_Ni_H1_min,Dihed_C1_N1_Ni_H1_max,Dihed_C1_N1_Ni_H1_mean
0,[H]C1=C(C(F)(F)F)C([H])=C([H])C(=C2N[C@@]([H])...,NGSCIQPXSPCNIO-AWEZNQCLSA-N,542.348569,553.008735,550.493550,29.330516,36.306605,34.344356,29.321253,33.171455,...,80.347226,-1.346289,3.191751,1.329373,-165.703650,36.631504,-31.528086,-167.700239,169.593983,-10.772839
1,[H]C1=C(C(F)(F)F)C([H])=C(S(=O)(=O)C([H])([H])...,RYNVXYBNTRCDQQ-LLVKDONJSA-N,556.626009,562.656106,560.505908,28.680899,28.820835,28.760296,31.770324,31.892324,...,80.818642,6.686289,9.378712,7.631905,-169.279626,-167.497045,-168.165134,171.407367,174.017534,172.165095
2,[H]C1=C(C(F)(F)F)C([H])=C(SC([H])([H])C([H])([...,WDGJHIYZYORLQS-LLVKDONJSA-N,534.547760,553.418087,548.988914,31.759090,31.850935,31.786836,28.803688,29.171068,...,81.510808,0.637924,7.065139,2.603857,-59.296394,-50.655177,-56.003500,97.351004,108.034239,103.050195
3,[H]C1=C(C#N)C([H])=C([H])C(=C2N[C@@]([H])(C(C(...,ZZYZJBMNVRAFBQ-LLVKDONJSA-N,459.480082,459.520074,459.500667,31.636695,31.647536,31.641956,27.853308,27.900019,...,81.008513,1.988838,2.496623,2.250208,-68.559634,-63.978255,-66.336406,110.065390,111.925355,111.022761
4,[H]C1=C(C(F)(F)F)C([H])=C([H])C(=C2N[C@@]([H])...,UITFYNBEZIIWCV-LURJTMIESA-N,426.787290,427.087289,426.923421,26.407240,26.540869,26.467877,27.295536,27.315048,...,81.832083,-0.057411,0.092376,0.024407,-179.338184,-178.268333,-178.852715,178.210941,179.297253,178.804314
5,[H]C1=C(C(F)(F)F)C([H])=C([H])C(=C2N[C@@]([H])...,ILENSZIBAAQNCZ-SNVBAGLBSA-N,462.677690,467.267554,466.777259,27.443947,28.672819,27.595336,30.090111,31.799691,...,81.887485,-0.441620,2.169861,-0.065941,-179.956019,179.288505,-19.474517,-179.810518,179.368307,19.349415
6,[H]C1=C(C(F)(F)F)C([H])=C([H])C(=C2N[C@@]([H])...,YZOJKEJLMYATFV-CYBMUJFWSA-N,509.228479,509.298481,509.263628,32.688578,32.772737,32.730835,27.537172,27.547618,...,82.030269,-0.646009,-0.533125,-0.589805,-37.408391,-24.728888,-31.041920,133.895707,138.734206,136.304761
7,[H]C1=C(C(F)(F)F)C([H])=C([H])C(=C2N[C@@]([H])...,UOIDAFBRAUSYGL-ZDUSSCGKSA-N,507.338541,537.208561,528.333801,26.457893,33.459604,27.523725,29.309033,41.108566,...,82.032779,-0.905985,0.906086,-0.486767,-179.969334,179.680555,-103.673637,171.406021,179.847116,178.564577
8,[H]C1=C(OC(F)(F)F)C([H])=C([H])C(=C2N[C@@]([H]...,KGZUJYDTOJDLJK-SNVBAGLBSA-N,494.097343,494.237329,494.164184,27.740768,31.685969,29.624551,27.695634,31.681830,...,81.312320,2.272652,2.418724,2.348977,-168.693369,-75.560632,-124.223682,105.858427,171.767497,140.296761
9,[H]C1=C(C(F)(F)F)C([H])=C([H])C(=C2N[C@@]([H])...,RJKNIIOSLLEQAJ-LBPRGKRZSA-N,511.618437,513.848416,512.458695,29.232561,29.467496,29.378972,33.075865,34.951397,...,80.176735,-0.788070,2.747292,0.544058,-165.238664,176.162127,47.521960,-176.513428,169.288915,-46.214751


In [42]:
total_df.to_csv("descriptor_stats.csv", index=False)